1. f"https://www.nongsaro.go.kr/portal/ps/psz/psza/contentMain.ps?menuId=PS03343&pageIndex={page}&pageSize=7&pageUnit=9&cntntsNo=&sUpperClCode=376004&sAreaNm={region}&sAreaCode=&sText="
2. 지역은 region을 통해 url에 넣기, 지역은 "전북특별자치도","전라남도", "경상북도", "경상남도"
3. 지역별 전체 page가 다름. 전북특별자치도 : 13, 전라남도 : 23, 경상북도 : 24, 경상남도 : 38
4. #contentForm > div.boardWrap > div.data_list > ul > li:nth-child(1) > a > div > dl > dt 농산물 이름 추출
5. #contentForm > div.boardWrap > div.data_list > ul > li:nth-child(1) > a > div > dl > dd 안에 지역 이름 존재
6. "경상남도 &gt; 고성군" -> &gt; 기준으로 앞의 것을 "도", 뒤의것을 "시군구"로 저장 (strip 적용해서 공백제거)
7. 각 페이지마다 리스트의 개수가 다를 수 있음. 예시로 #contentForm > div.boardWrap > div.data_list > ul > li:nth-child(8) > a > span 8개밖에 없는것도 있음. 
8. 결과는 데이터프레임으로 전체 저장
9. 각 region, 각 페이지를 진행 한 이후 csv파일을 누적생성. 이름은 region_page.csv, 인코딩 : utf-8-sig
10. time.sleep함수는 사용하지 말고 webdriver를 통해 페이지 로딩 기달리기

In [1]:
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# 1) 기본 설정
regions = {
    "전북특별자치도": 13,
    "전라남도": 23,
    "경상북도": 24,
    "경상남도": 38
}
base_url = "https://www.nongsaro.go.kr/portal/ps/psz/psza/contentMain.ps"
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

# 2) 드라이버 초기화 (예: Chrome)
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

all_data = []

for region, max_page in regions.items():
    for page in range(1, max_page+1):
        # 3) URL 조립 및 페이지 오픈
        params = (
            f"?menuId=PS03343"
            f"&pageIndex={page}"
            f"&pageSize=7"
            f"&pageUnit=9"
            f"&cntntsNo="
            f"&sUpperClCode=376004"
            f"&sAreaNm={region}"
            f"&sAreaCode="
            f"&sText="
        )
        driver.get(base_url + params)

        # 4) 리스트 로딩 대기
        wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR,
            "#contentForm .data_list ul li"
        )))

        # 5) 현재 페이지의 모든 li 요소 순회
        items = driver.find_elements(By.CSS_SELECTOR, "#contentForm .data_list ul li")
        records = []
        for li in items:
            # 상품명 추출
            name = li.find_element(By.CSS_SELECTOR, "a div dl dt").text.strip()
            # 지역 정보(dd) 추출 후 앞/뒤 분리
            full_loc = li.find_element(By.CSS_SELECTOR, "a div dl dd").text.strip()
            # "경상남도 > 고성군" 형태를 '>'로 분리
            parts = [p.strip() for p in full_loc.split(">")]
            do = parts[0] if len(parts) >= 1 else ""
            sigungu = parts[1] if len(parts) >= 2 else ""

            records.append({
                "상품명": name,
                "도": do,
                "시군구": sigungu,
                "원래지역": region,
                "페이지": page,
            })

        # 6) 페이지별로 CSV 저장 (utf-8-sig)
        df_page = pd.DataFrame(records)
        csv_path = os.path.join(output_dir, f"{region}_{page}.csv")
        df_page.to_csv(csv_path, index=False, encoding="utf-8-sig")

        # 7) 전체 데이터에도 누적
        all_data.extend(records)

# 8) 전체 결과를 하나의 DataFrame으로
df_all = pd.DataFrame(all_data)
df_all.to_csv(os.path.join(output_dir, "all_regions_all_pages.csv"),
              index=False, encoding="utf-8-sig")

driver.quit()


In [7]:
df_all.head()

,상품명,도,시군구,원래지역,페이지
0,부안해풍마늘,전북특별자치도,부안군,전북특별자치도,1
1,부안버섯,전북특별자치도,부안군,전북특별자치도,1
2,부안해풍양파,전북특별자치도,부안군,전북특별자치도,1
3,레드향,전북특별자치도,부안군,전북특별자치도,1
4,멜론,전북특별자치도,부안군,전북특별자치도,1


In [9]:
result=df_all.drop(columns=["원래지역","페이지"])

In [10]:
result.to_csv(os.path.join(output_dir, "영남호남_지역특산물.csv"),
              index=False, encoding="utf-8-sig")